# 🦟 พยากรณ์จำนวนผู้ป่วยโรคติดต่อนำโดยแมลง (Vector-Borne Disease Forecasting)
## คลาสเรียนรู้ Machine Learning อย่างง่าย สำหรับการพยากรณ์ข้อมูลอนุกรมเวลา (Time-Series Forecasting)
### 📊 แสดงผลแผนภูมิด้วย Plotly (Interactive Chart) เพื่อรองรับภาษาไทย 100%

---

### 📋 วัตถุประสงค์
1. เรียนรู้วิธีการนำข้อมูลดิบมารวมกันและจัดการโครงสร้างข้อมูลสำหรับอนุกรมเวลา (Time-Series)
2. เรียนรู้การวิเคราะห์สร้างตัวแปรต้นเพื่อช่วยในการทำนายอดีต (Feature Engineering / Lags)
3. ฝึกสอนโมเดล Machine Learning อย่างง่าย (Linear Regression & Random Forest Regressor) เพื่อทำนายและเปรียบเทียบผลลัพธ์
4. ทำนายอนาคต (Future Forecasting) 6 เดือนข้างหน้าด้วยโมเดลแบบวนซ้ำ (Recursive Forecasting)

## 1. Setup & Import Libraries 📦

In [28]:
import pandas as pd
import numpy as np
import glob
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook_connected'
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("✅ Import สำเร็จ และพร้อมใช้งาน (ใช้ Plotly สำหรับการวาดกราฟ)")

✅ Import สำเร็จ และพร้อมใช้งาน (ใช้ Plotly สำหรับการวาดกราฟ)


## 2. Load and Merge Datasets 📁
ทำการโหลดข้อมูลจำนวนผู้ประสบโรคติดต่อนำโดยแมลงตั้งแต่ปี 2015 ถึงปี 2025 มารวมกัน

In [29]:
# ค้นหาไฟล์ข้อมูลรายปีทั้งหมด
data_dir = os.path.join("Diseases", "กลุ่มโรคติดต่อนำโดยแมลง")
file_pattern = os.path.join(data_dir, "กลุ่มโรคติดต่อนำโดยแมลง_ปี_*.csv")
all_files = glob.glob(file_pattern)
all_files.sort()  # เรียงลำดับปี

print(f"พบไฟล์ข้อมูลรายปีทั้งหมด {len(all_files)} ไฟล์:")
for f in all_files:
    print(f" - {os.path.basename(f)}")

# โหลดข้อมูลจากทุกไฟล์และนำมารวมกัน (Concat)
df_list = []
for file_path in all_files:
    df_temp = pd.read_csv(file_path, encoding='utf-8')
    df_list.append(df_temp)

df_all = pd.concat(df_list, ignore_index=True)
print(f"\n📊 รวมข้อมูลเสร็จสมบูรณ์! จำนวนทั้งหมด {len(df_all):,} แถว")
display(df_all.head(5))

พบไฟล์ข้อมูลรายปีทั้งหมด 11 ไฟล์:
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2015.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2016.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2017.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2018.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2019.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2020.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2021.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2022.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2023.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2024.csv
 - กลุ่มโรคติดต่อนำโดยแมลง_ปี_2025.csv

📊 รวมข้อมูลเสร็จสมบูรณ์! จำนวนทั้งหมด 6,480 แถว


,year,month,event_name,disease_group,province,province_name
0,2015,1,DF/D.H.F./DSS,แมลง,สระบุรี,NaN
1,2015,1,DF/D.H.F./DSS,แมลง,สระบุรี,NaN
2,2015,1,DF/D.H.F./DSS,แมลง,สระบุรี,NaN
3,2015,1,Lymphatic filariasis,แมลง,ระยอง,NaN
4,2015,2,DF/D.H.F./DSS,แมลง,ชลบุรี,NaN


## 3. Data Aggregation & Time-Series แปลงเป็นอนุกรมเวลา 📈
นับจำนวนผู้ป่วยรายเดือน (โดย 1 แถวของข้อมูลดิบหมายถึงผู้ป่วย 1 ราย)

In [30]:
# ดูประเภทโรคหลักๆ ที่มี
print("📋 จำนวนโรคในระบบข้อมูล:")
print(df_all['event_name'].value_counts())

# รวมจำนวนผู้ป่วย (นับจำนวนแถว) แยกรายปีและรายเดือน
monthly_cases = df_all.groupby(['year', 'month']).size().reset_index(name='cases')

# สร้างคอลัมน์ประเภท Date เพื่อสะดวกต่อการแสดงผลและแบ่งชุดข้อมูล
monthly_cases['date'] = pd.to_datetime(
    monthly_cases['year'].astype(str) + '-' + monthly_cases['month'].astype(str).str.zfill(2) + '-01'
)
monthly_cases = monthly_cases.sort_values('date').reset_index(drop=True)

print(f"\n📈 ข้อมูลอนุกรมเวลาพร้อมใช้! มีจำนวนเดือนทั้งหมด: {len(monthly_cases)} เดือน")
display(monthly_cases.head(10))

# พล็อตแนวโน้มข้อมูลจริงโดยใช้ Plotly
fig = px.line(
    monthly_cases, 
    x='date', 
    y='cases', 
    markers=True, 
    title='แนวโน้มจำนวนผู้ป่วยโรคติดต่อนำโดยแมลงรายเดือน (ปี 2015 - 2025)'
)
fig.update_traces(line=dict(color='#2980b9', width=2), marker=dict(size=6))
fig.update_layout(
    xaxis_title='ปี-เดือน',
    yaxis_title='จำนวนผู้ป่วย (ราย)',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

📋 จำนวนโรคในระบบข้อมูล:
event_name
DF/D.H.F./DSS                                         2406
Zika                                                  1739
Chikungunya fever                                      955
Malaria                                                698
DF, DHF, DSS                                           365
Scrub typhus                                            64
Leishmaniasis                                           57
Lymphatic filariasis                                    56
Scrub Typhus                                            49
อื่นๆ ระบุ                                              44
Severe Fever with Thrombocytopenia Syndrome (SFTS)      18
Filariasis                                              17
Scabies                                                  6
Animal envenomation                                      4
Melioidosis                                              1
Fever/Pyrexia/PUO/Febrile                                1
Name: count, dtype: i

,year,month,cases,date
0,2015,1,7,2015-01-01
1,2015,2,5,2015-02-01
2,2015,3,7,2015-03-01
3,2015,4,7,2015-04-01
4,2015,5,11,2015-05-01
5,2015,6,16,2015-06-01
6,2015,7,33,2015-07-01
7,2015,8,36,2015-08-01
8,2015,9,27,2015-09-01
9,2015,10,26,2015-10-01


## 4. Feature Engineering: สร้างค่า Lag 🛠️
ในการพยากรณ์อนุกรมเวลา ตัวแปรต้นที่ดีที่สุดมักจะเป็น **"ค่าผู้ป่วยในอดีต (Lag)"** เช่น:
- `lag_1`: จำนวนผู้ป่วยของเดือนก่อนหน้า
- `lag_2`: จำนวนผู้ป่วยย้อนหลังไป 2 เดือน
- `lag_12`: จำนวนผู้ป่วยย้อนหลังไป 12 เดือน (สะท้อนพฤติกรรมฤดูกาล/ปีที่แล้ว ซึ่งสำคัญมากเพราะโรคจากแมลงมักระบาดช่วงหน้าฝนเหมือนกันทุกปี)

In [31]:
# สร้าง Lag Features
monthly_cases['lag_1'] = monthly_cases['cases'].shift(1)
monthly_cases['lag_2'] = monthly_cases['cases'].shift(2)
monthly_cases['lag_12'] = monthly_cases['cases'].shift(12)  # ฤดูกาลรายปี

# ลบแถวแรกๆ ที่ไม่มีประวัติย้อนหลังพอ (ค่าเป็น NaN)
df_features = monthly_cases.dropna().reset_index(drop=True)

# เลือกตัวแปรต้น (X) และตัวแปรตามที่ต้องการทำนาย (y)
features = ['month', 'lag_1', 'lag_2', 'lag_12']
X = df_features[features]
y = df_features['cases']

print("📊 หน้าตาข้อมูลสำหรับการสร้างโมเดล (มี Features และ Target):")
display(df_features[['date', 'cases'] + features].head(10))

📊 หน้าตาข้อมูลสำหรับการสร้างโมเดล (มี Features และ Target):


,date,cases,month,lag_1,lag_2,lag_12
0,2016-01-01,22,1,13.0,21.0,7.0
1,2016-02-01,17,2,22.0,13.0,5.0
2,2016-03-01,18,3,17.0,22.0,7.0
3,2016-04-01,10,4,18.0,17.0,7.0
4,2016-05-01,6,5,10.0,18.0,11.0
5,2016-06-01,15,6,6.0,10.0,16.0
6,2016-07-01,22,7,15.0,6.0,33.0
7,2016-08-01,13,8,22.0,15.0,36.0
8,2016-09-01,46,9,13.0,22.0,27.0
9,2016-10-01,38,10,46.0,13.0,26.0


## 5. แบ่งชุดข้อมูล Train & Test 🕒
สำหรับข้อมูลอนุกรมเวลา เรา**ไม่สามารถ**ใช้การสุ่มข้อมูล (Random Split) ได้ เพราะจะทำให้ข้อมูลในอนาคตหลุดไปอยู่ใน Train Set (Data Leakage) 
ดังนั้นเราจะตัดแบ่งตามแกนเวลาตรงๆ โดยเลือกประวัติส่วนใหญ่มา Train และใช้เดือนช่วงท้ายสุดของชุดข้อมูลมา Test

In [32]:
# นำข้อมูลช่วงท้ายสุด 18 เดือนมาใช้เป็น Test Set สำหรับประเมินความแม่นยำของโมเดล
test_size = 18
train_idx = len(df_features) - test_size

X_train, X_test = X.iloc[:train_idx], X.iloc[train_idx:]
y_train, y_test = y.iloc[:train_idx], y.iloc[train_idx:]

dates_train = df_features['date'].iloc[:train_idx]
dates_test = df_features['date'].iloc[train_idx:]

print(f"✅ ข้อมูล Train: {len(X_train)} เดือน (ใช้ประวัติย้อนหลังสร้างโมเดล)")
print(f"✅ ข้อมูล Test:  {len(X_test)} เดือน (สำหรับทดสอบจริง ตั้งแต่ {dates_test.min().strftime('%Y-%m')} ถึง {dates_test.max().strftime('%Y-%m')})")

✅ ข้อมูล Train: 102 เดือน (ใช้ประวัติย้อนหลังสร้างโมเดล)
✅ ข้อมูล Test:  18 เดือน (สำหรับทดสอบจริง ตั้งแต่ 2024-07 ถึง 2025-12)


## 6. สร้างและฝึกสอนโมเดล (Model Training) 🤖
เราจะสร้าง 2 โมเดลยอดนิยมมาเรียนรู้ข้อมูล:
1. **Linear Regression** (แบบจำลองสมการเส้นตรงเชิงเส้นอย่างง่าย)
2. **Random Forest Regressor** (แบบจำลองโครงสร้างต้นไม้ตัดสินใจหลายร้อยต้น - แม่นยำกับข้อมูลไม่เชิงเส้น)

In [33]:
# 1. ฝึกสอนโมเดล Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

# 2. ฝึกสอนโมเดล Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# ฟังก์ชันประเมินความถูกต้องของแบบจำลอง
def evaluate_model(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"📊 ประสิทธิภาพของโมเดล [{name}]:")
    print(f"  - ค่าความคลาดเคลื่อนกำลังสองเฉลี่ยสะสม (RMSE): {rmse:.2f} ราย")
    print(f"  - ค่าความคลาดเคลื่อนสมบูรณ์เฉลี่ย (MAE): {mae:.2f} ราย")
    print(f"  - ค่าความสัมพันธ์ R² Score (ใกล้ 1 ยิ่งดี): {r2:.4f}")
    print("-" * 45)
    return rmse, mae, r2

lr_metrics = evaluate_model(y_test, lr_pred, "Linear Regression")
rf_metrics = evaluate_model(y_test, rf_pred, "Random Forest")

📊 ประสิทธิภาพของโมเดล [Linear Regression]:
  - ค่าความคลาดเคลื่อนกำลังสองเฉลี่ยสะสม (RMSE): 26.36 ราย
  - ค่าความคลาดเคลื่อนสมบูรณ์เฉลี่ย (MAE): 20.58 ราย
  - ค่าความสัมพันธ์ R² Score (ใกล้ 1 ยิ่งดี): 0.3559
---------------------------------------------
📊 ประสิทธิภาพของโมเดล [Random Forest]:
  - ค่าความคลาดเคลื่อนกำลังสองเฉลี่ยสะสม (RMSE): 27.85 ราย
  - ค่าความคลาดเคลื่อนสมบูรณ์เฉลี่ย (MAE): 18.97 ราย
  - ค่าความสัมพันธ์ R² Score (ใกล้ 1 ยิ่งดี): 0.2806
---------------------------------------------


## 7. เปรียบเทียบผลการทำนายจริง (Plotly Interactive) 📊
แสดงผลภาพเปรียบเทียบข้อมูลจริงและผลทำนายแบบโต้ตอบได้ (Interactive) เพื่อความยืดหยุ่นในการดูช่วงเวลาต่างๆ

In [34]:
fig = go.Figure()

# ข้อมูลจริงทั้งหมด
fig.add_trace(go.Scatter(
    x=df_features['date'], 
    y=df_features['cases'], 
    name='ข้อมูลจริงในอดีต (Actual Cases)', 
    line=dict(color='#2c3e50', width=2.5)
))

# ผลทำนาย Linear Regression
fig.add_trace(go.Scatter(
    x=dates_test, 
    y=lr_pred, 
    name='ทำนายโดย Linear Regression', 
    line=dict(color='#e74c3c', width=2, dash='dash')
))

# ผลทำนาย Random Forest
fig.add_trace(go.Scatter(
    x=dates_test, 
    y=rf_pred, 
    name='ทำนายโดย Random Forest', 
    line=dict(color='#27ae60', width=2, dash='dot')
))

# แรเงาช่วง Test Period
fig.add_vrect(
    x0=dates_test.min(), 
    x1=dates_test.max(), 
    fillcolor='#f1c40f', 
    opacity=0.15, 
    layer='below', 
    line_width=0, 
    annotation_text='ช่วงที่ทดสอบทำนาย (Test Period)', 
    annotation_position='top left'
)

fig.update_layout(
    title='การเปรียบเทียบผลลัพธ์ของโมเดลการพยากรณ์จำนวนเคสผู้ประสบโรค (Actual vs Prediction)',
    xaxis_title='ปี-เดือน',
    yaxis_title='จำนวนผู้ป่วย (ราย)',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

## 8. ความสำคัญของฟีเจอร์ต่างๆ (Feature Importance) 🔍
วิเคราะห์ความสำคัญของข้อมูลแต่ละส่วนที่ส่งผลต่อการทำนายโรค โดยใช้ฟังก์ชันของ Random Forest วาดแผนภูมิแท่งผ่าน Plotly

In [35]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

# สร้าง DataFrame เพื่อวาดกราฟ
imp_df = pd.DataFrame({
    'Feature': [features[i] for i in indices],
    'Importance': importances[indices]
})

fig = px.bar(
    imp_df, 
    x='Feature', 
    y='Importance', 
    title='ตัวแปรที่ส่งผลต่อการทำนายของโมเดลมากที่สุด (Feature Importance)',
    color='Importance',
    color_continuous_scale=['#9575CD', '#311B92']
)
fig.update_layout(
    xaxis_title='ฟีเจอร์/ตัวแปรต้น',
    yaxis_title='ค่าน้ำหนักความสำคัญ (Relative Importance)',
    template='plotly_white',
    showlegend=False
)
fig.show()

## 9. การพยากรณ์อนาคตล่วงหน้า 6 เดือน (Future Forecasting) 🔮
ใช้โมเดลที่ฝึกสำเร็จในการพยากรณ์แบบ **Recursive Multi-step Forecasting** (การนำเอาผลทำนายของขั้นตอนก่อนหน้ามาใช้เป็นค่า Lag ในขั้นถัดไปวนไปจนครบกำหนด)

In [36]:
# ดึงจุดเริ่มต้นจากเดือนล่าสุดในฐานข้อมูลเพื่อพยากรณ์ต่อ
last_row = df_features.iloc[-1]
current_date = last_row['date']

# พยากรณ์ล่วงหน้า 6 เดือน
future_forecast = []
forecast_dates = []

# โคลนประวัติข้อมูลเพื่อใช้อ้างอิงการลากย้อนหลัง
history_cases = list(df_features['cases'].values)
history_dates = list(df_features['date'].values)

# ทำนายลูป 6 เดือนข้างหน้า
for step in range(1, 7):
    next_date = current_date + pd.DateOffset(months=step)
    next_month = next_date.month
    
    # ดึงค่า Lag 1, Lag 2, Lag 12 จากประวัติสะสม
    lag_1 = history_cases[-1]
    lag_2 = history_cases[-2]
    lag_12 = history_cases[-12]
    
    # สร้าง DataFrame สำหรับโมเดลทำนาย
    X_step = pd.DataFrame([[next_month, lag_1, lag_2, lag_12]], columns=features)
    
    # ใช้ Random Forest พยากรณ์ (เนื่องจากค่า R² Score สูงกว่าและเสถียรกว่า)
    pred_step = rf_model.predict(X_step)[0]
    if pred_step < 0:
        pred_step = 0  # จำนวนเคสติดเชื้อต้องไม่ติดลบ
        
    # บันทึกผลลัพธ์
    future_forecast.append(pred_step)
    forecast_dates.append(next_date)
    history_cases.append(pred_step)  # นำค่าพยากรณ์ใส่ในประวัติเพื่อใช้อ้างอิงขั้นตอนถัดไป

# แสดงตารางผลพยากรณ์อนาคตล่วงหน้า
future_df = pd.DataFrame({
    'เดือนพยากรณ์ (Date)': [d.strftime('%Y-%m') for d in forecast_dates],
    'จำนวนเคสพยากรณ์ (Predicted Cases)': [round(v) for v in future_forecast]
})
print("🎯 ผลการทำนายพยากรณ์จำนวนเคสล่วงหน้า 6 เดือนข้างหน้า:")
display(future_df)

# พล็อตแผนภูมิพยากรณ์อนาคตแบบ Interactive ด้วย Plotly
fig = go.Figure()

# ข้อมูลย้อนหลัง 2 ปีล่าสุด
history_subset = df_features.tail(24)
fig.add_trace(go.Scatter(
    x=history_subset['date'],
    y=history_subset['cases'],
    mode='lines+markers',
    name='เคสจริงย้อนหลัง (2 ปีล่าสุด)',
    line=dict(color='#2c3e50', width=2)
))

# ผลทำนายล่วงหน้า
fig.add_trace(go.Scatter(
    x=forecast_dates,
    y=future_forecast,
    mode='lines+markers',
    name='ทำนายล่วงหน้า 6 เดือน (Forecast)',
    line=dict(color='#e67e22', width=2.5, dash='dash'),
    marker=dict(symbol='x', size=8)
))

fig.update_layout(
    title='การคาดการณ์จำนวนผู้ป่วยโรคติดต่อนำโดยแมลงล่วงหน้า 6 เดือนข้างหน้า (Plotly Forecast)',
    xaxis_title='ปี-เดือน',
    yaxis_title='จำนวนผู้ป่วย (ราย)',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

🎯 ผลการทำนายพยากรณ์จำนวนเคสล่วงหน้า 6 เดือนข้างหน้า:


,เดือนพยากรณ์ (Date),จำนวนเคสพยากรณ์ (Predicted Cases)
0,2026-01,18
1,2026-02,13
2,2026-03,15
3,2026-04,17
4,2026-05,17
5,2026-06,13
